# Tech Challenge Fase 1

## Bibliotecas

In [57]:
import pandas as pd
import kagglehub
import os
import openpyxl
import numpy as np

## Dataset

In [58]:
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print(os.listdir(path))

['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


In [59]:
customers = pd.read_csv(os.path.join(path, "olist_customers_dataset.csv")) # Cliente
orders = pd.read_csv(os.path.join(path, "olist_orders_dataset.csv")) # Pedidos
order_items = pd.read_csv(os.path.join(path, "olist_order_items_dataset.csv")) # Pedidos - Items
payments = pd.read_csv(os.path.join(path, "olist_order_payments_dataset.csv")) # Pagamentos
reviews = pd.read_csv(os.path.join(path, "olist_order_reviews_dataset.csv")) # NPS
products = pd.read_csv(os.path.join(path, "olist_products_dataset.csv")) # Produtos
sellers = pd.read_csv(os.path.join(path, "olist_sellers_dataset.csv")) # Vendedores/Lojas
geolocation = pd.read_csv(os.path.join(path, "olist_geolocation_dataset.csv")) # Geo
category_translation = pd.read_csv(os.path.join(path, "product_category_name_translation.csv")) # Traducao produtos.

## Tratando dataset

In [60]:
cols_orders = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

cols_reviews = ["review_creation_date", "review_answer_timestamp"]

order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"]) # .dt.date

for col in cols_orders:
    orders[col] = pd.to_datetime(orders[col]) # .dt.date

for col in cols_reviews:
    reviews[col] = pd.to_datetime(reviews[col]) # .dt.date

## Criando colunas essenciais (gráficos e análises)

In [61]:
# compras por ano e mês (separados)
orders["purchase_year"] = orders["order_purchase_timestamp"].dt.year
orders["purchase_month"] = orders["order_purchase_timestamp"].dt.month

# para ordenar no bi
orders["purchase_month_date"] = orders["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()

# compras por ano-mês (juntos)
orders["purchase_year_month"] = orders["order_purchase_timestamp"].dt.strftime("%Y-%m")

# qtd dias para o cliente receber. (data de entrega menos data da compra)
orders["delivery_days"] = (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.days

# qtd dias para aprovar pedido/pagamento (data da aprovação menos a data da compra)
orders["approval_days"] = (orders["order_approved_at"] - orders["order_purchase_timestamp"]).dt.days

# qtd dias para postar na transportadora (data de entrega na transportadora menos data da aprovação)
orders["carrier_days"] = (orders["order_delivered_carrier_date"] - orders["order_approved_at"]).dt.days

# prazo prometido ao cliente  (data esperada da entrega menos data da compra.)
orders["estimated_delivery_days"] = (orders["order_estimated_delivery_date"] - orders["order_purchase_timestamp"]).dt.days

# calculando atraso (data de entrega menos data estimada da entrega)
orders["delay_days"] = (orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]).dt.days

# flag de atraso (se o delay for maior do que zero, que dizer que atrasou, se não, não)
orders["is_late"] = (orders["delay_days"] > 0).astype(int)

orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,purchase_year,purchase_month,purchase_month_date,purchase_year_month,delivery_days,approval_days,carrier_days,estimated_delivery_days,delay_days,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017,10,2017-10-01,2017-10,8.0,0.0,2.0,15,-8.0,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2018,7,2018-07-01,2018-07,13.0,1.0,0.0,19,-6.0,0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,2018,8,2018-08-01,2018-08,9.0,0.0,0.0,26,-18.0,0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,2017,11,2017-11-01,2017-11,13.0,0.0,3.0,26,-13.0,0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2018,2,2018-02-01,2018-02,2.0,0.0,0.0,12,-10.0,0


In [62]:
# valor total do pedido (preço + frete)

order_items["item_total"] = order_items["price"] + order_items["freight_value"]

In [63]:
# conferindo se tem mais de um pagamento por order_id

payments["order_id"].value_counts()

order_id
fa65dad1b0e818e3ccc5cb0e39231352    29
ccf804e764ed5650cd8759557269dc13    26
285c2e15bebd4ac83635ccc563dc71f4    22
895ab968e7bb0d5659d16cd74cd1650c    21
ee9ca989fc93ba09a6eddc250ce01742    19
                                    ..
0406037ad97740d563a178ecc7a2075c     1
7b905861d7c825891d6347454ea7863f     1
32609bbb3dd69b3c066a6860554a77bf     1
b8b61059626efa996a60be9bb9320e10     1
28bbae6599b09d39ca406b747b6632b1     1
Name: count, Length: 99440, dtype: int64

In [64]:
reviews["review_is_good"] = (reviews["review_score"] >= 4).astype(int)

reviews["review_is_bad"] = (reviews["review_score"] <= 2).astype(int)

reviews["review_response_days"] = (
    reviews["review_answer_timestamp"] - reviews["review_creation_date"]
).dt.days


reviews['review_comment_message'] = reviews['review_comment_message'].str.upper()
reviews["review_creation_year_month"] = reviews["review_creation_date"].dt.to_period("M")

reviews_bad = reviews[reviews["review_score"] < 2]

## Conferência

In [65]:
# total de linhas (total de reviews)
total_reviews = reviews["review_id"].count()

# total de review_id distintos
unique_reviews = reviews["review_id"].nunique()

print("TOTAL (linhas):", total_reviews)
print("DISTINTOS (review_id):", unique_reviews)

TOTAL (linhas): 99224
DISTINTOS (review_id): 98410


In [66]:
reviews_bad = reviews[reviews["review_score"] < 2]

total_reviews_bad = reviews_bad["review_id"].count()
unique_reviews_bad = reviews_bad["review_id"].nunique()

print("TOTAL (linhas) score < 2:", total_reviews_bad)
print("DISTINTOS (review_id) score < 2:", unique_reviews_bad)

TOTAL (linhas) score < 2: 11424
DISTINTOS (review_id) score < 2: 11282


In [67]:
reviews_good = reviews[reviews["review_score"] > 2]

total_reviews_good = reviews_good["review_id"].count()
unique_reviews_good = reviews_good["review_id"].nunique()

print("TOTAL (linhas) score > 2:", total_reviews_good)
print("DISTINTOS (review_id) score > 2:", unique_reviews_good)

TOTAL (linhas) score > 2: 84649
DISTINTOS (review_id) score > 2: 84014


## Agregando (group by)

In [68]:
order_items_summary = order_items.groupby("order_id").agg(
    items_count=("order_item_id", "count"),
    items_total_price=("price", "sum"),
    items_total_freight=("freight_value", "sum"),
    items_total_value=("item_total", "sum")
).reset_index()

order_items_summary.head()

,order_id,items_count,items_total_price,items_total_freight,items_total_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,218.04


In [69]:
items_prod = order_items.merge(
    products[["product_id", "product_category_name"]],
    on="product_id",
    how="left"
)

cat_order = (
    items_prod.groupby(["order_id", "product_category_name"])["item_total"]
    .sum()
    .reset_index()
)

cat_main = (
    cat_order.sort_values(["order_id", "item_total"], ascending=[True, False])
    .drop_duplicates("order_id")
)

cat_main = cat_main[["order_id", "product_category_name"]]

In [70]:
seller_order = (
    order_items.groupby(["order_id", "seller_id"])["item_total"]
    .sum()
    .reset_index()
)

seller_main = seller_order.sort_values(["order_id", "item_total"], ascending=[True, False]) \
                          .drop_duplicates("order_id")

seller_main = seller_main[["order_id", "seller_id"]]

seller_main.head()

,order_id,seller_id
0,00010242fe8c5a6d1ba2dd792cb16214,48436dade18ac8b2bce089ec2a041202
1,00018f77f2f0320c557190d7a144bdd3,dd7ddc04e1b6c2c614352b383efe2d36
2,000229ec398224ef6ca0657da4fc703e,5b51032eddd242adc84c38acab88f23d
3,00024acbcdf0a6daa1e931b038114c75,9d7a1d34a5052409006425275ba1c2b4
4,00042b26cf59d7ce69dfabb4e55b4fd9,df560393f3a51e74553ab94004ba5c87


In [71]:
reviews_summary = reviews.groupby("order_id").agg(
    review_score_mean=("review_score", "mean"),
    review_is_good_rate=("review_is_good", "mean"),
    review_is_bad_rate=("review_is_bad", "mean"),
    review_response_days_mean=("review_response_days", "mean")
).reset_index()

reviews_summary.head()

,order_id,review_score_mean,review_is_good_rate,review_is_bad_rate,review_response_days_mean
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,1.0,0.0,1.0
1,00018f77f2f0320c557190d7a144bdd3,4.0,1.0,0.0,2.0
2,000229ec398224ef6ca0657da4fc703e,5.0,1.0,0.0,0.0
3,00024acbcdf0a6daa1e931b038114c75,4.0,1.0,0.0,0.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,1.0,0.0,1.0


In [72]:
# agrupando os pedidos e calculando o valor total de pagamento, média de parcelas e a principal forma de pagamento.

payments_summary = payments.groupby("order_id").agg(
    payment_value_total=("payment_value", "sum"),
    installments_mean=("payment_installments", "mean"),
    payment_type_main=("payment_type", "first")
).reset_index()

payments_summary.head()

,order_id,payment_value_total,installments_mean,payment_type_main
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2.0,credit_card
1,00018f77f2f0320c557190d7a144bdd3,259.83,3.0,credit_card
2,000229ec398224ef6ca0657da4fc703e,216.87,5.0,credit_card
3,00024acbcdf0a6daa1e931b038114c75,25.78,2.0,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3.0,credit_card


## Join

In [73]:
fact_orders = (
    orders
    .merge(customers, on="customer_id", how="left")
    .merge(order_items_summary, on="order_id", how="left")
    .merge(payments_summary, on="order_id", how="left")
    .merge(reviews_summary, on="order_id", how="left")
    .merge(seller_main, on="order_id", how="left")
    .merge(cat_main, on="order_id", how= "left")
)

In [74]:
fact_orders.shape[0] == fact_orders["order_id"].nunique()

True

In [75]:
fact_orders.head(10)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,purchase_year,purchase_month,...,items_total_value,payment_value_total,installments_mean,payment_type_main,review_score_mean,review_is_good_rate,review_is_bad_rate,review_response_days_mean,seller_id,product_category_name
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017,10,...,38.71,38.71,1.0,credit_card,4.0,1.0,0.0,1.0,3504c0cb71d7fa48d967e0e4c94d59d9,utilidades_domesticas
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2018,7,...,141.46,141.46,1.0,boleto,4.0,1.0,0.0,0.0,289cdb325fb7e7f891c38608bf9e0962,perfumaria
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,2018,8,...,179.12,179.12,3.0,credit_card,5.0,1.0,0.0,4.0,4869f7a5dfa277a7dca6462dcf3b52b2,automotivo
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,2017,11,...,72.20,72.20,1.0,credit_card,5.0,1.0,0.0,2.0,66922902710d126a0e7d26b0e3805106,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2018,2,...,28.62,28.62,1.0,credit_card,5.0,1.0,0.0,1.0,2c9e548be18521d1c43cde1c582c6de8,papelaria
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01,2017,7,...,175.26,175.26,6.0,credit_card,4.0,1.0,0.0,0.0,8581055ce74af1daba164fdbd55a40de,automotivo
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaT,NaT,2017-05-09,2017,4,...,65.95,65.95,1.0,credit_card,2.0,0.0,1.0,0.0,dc8798cbf453b7e0f98745e396cc5616,NaN
7,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07,2017,5,...,75.16,75.16,3.0,credit_card,5.0,1.0,0.0,1.0,16090f2ca825584b5a147ab24aa30c86,automotivo
8,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06,2017,1,...,35.95,35.95,1.0,boleto,1.0,0.0,1.0,2.0,63b9ae557efed31d1f7687917d248a8d,moveis_decoracao
9,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23,2017,7,...,169.76,169.76,1.0,voucher,5.0,1.0,0.0,1.0,7c67e1448b00f6e969d365cea6b010ab,moveis_escritorio


In [76]:
fact_orders.to_csv("fact_orders_2.csv", index=False, sep=";", decimal=",")